In [4]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126 -U

Looking in indexes: https://download.pytorch.org/whl/cu126


In [66]:
from torch import nn
import torch
from typing import NamedTuple
from torch.func import rearrange


In [69]:
class BackboneResult(NamedTuple):
    c3: torch.Tensor
    c4: torch.Tensor
    c5: torch.Tensor

class PyramidResult(NamedTuple):
    p3: torch.Tensor
    p4: torch.Tensor
    p5: torch.Tensor

class convolutionalBlock(nn.Module):
    def __init__(self, input_layers, output_layers, conv_num=2, kernel_size=2, stride=2, padding=0):
      super().__init__()
      layers = []

      for i in range(conv_num):
          layers.extend([
              nn.Conv2d(
                  input_layers if i == 0 else output_layers,
                  output_layers,
                  kernel_size,
                  stride=stride,
                  padding=padding
              ),
              nn.BatchNorm2d(output_layers),
              nn.ReLU()
          ])

      self.cnv = nn.Sequential(*layers)

    def forward(self, x):
      x = self.cnv(x)

      return x

class Backbone(nn.Module):
    def __init__(self, input_layers=3):
        super().__init__()
        self.cnv1 = convolutionalBlock(input_layers, 128, 3)
        self.cnv2 = convolutionalBlock(128, 256, 1)
        self.cnv3 = convolutionalBlock(256, 512, 1)

    def forward(self, x):
        x_1 = self.cnv1(x)
        x_2 = self.cnv2(x_1)
        x_3 = self.cnv3(x_2)

        return BackboneResult(x_1, x_2, x_3)

class fusionModule(nn.Module):
    def __init__(self, in_channels=64):
        super().__init__()
        self.upscaler = nn.Upsample(scale_factor=2, mode="nearest")
        self.cnv_block = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU()
        )

    def forward(self, c5, c4):
        x = self.upscaler(c5) + c4
        x = self.cnv_block(x)

        return x

class Neck(nn.Module):
    def __init__(
        self,
        first_layers=128,
        second_layers=256,
        third_layers=512,
        feature_layers=64
    ):
        super().__init__()

        self.fusion2 = fusionModule(feature_layers)
        self.fusion3 = fusionModule(feature_layers)

        self.c5_cnv = nn.Conv2d(
            third_layers,
            feature_layers,
            kernel_size=1,
            stride=1
        )
        self.c4_cnv = nn.Conv2d(
            second_layers,
            feature_layers,
            kernel_size=1,
            stride=1
        )
        self.c3_cnv = nn.Conv2d(
            first_layers,
            feature_layers,
            kernel_size=1,
            stride=1
        )

    def forward(self, x):
        x_c5 = self.c5_cnv(x.c5)
        x_c4 = self.c4_cnv(x.c4)
        x_c3 = self.c3_cnv(x.c3)

        x_c4 = self.fusion2(x_c5, x_c4)
        x_c3 = self.fusion3(x_c4, x_c3)

        return PyramidResult(x_c3, x_c4, x_c5)

class Head(nn.Module):
    def __init__(self, feature_layers, num_anchors=3, num_classes=3):
        super().__init__()

        self.layers_num = num_anchors * (num_classes + 1 + 4)
        self.cnv = nn.Conv2d(
            feature_layers,
            self.layers_num,
            kernel_size=3,
            padding=1,
            stride=1
        )

    def forward(self, x):
        return self.cnv(x)

class FPN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = Backbone()
        self.neck = Neck()

        self.hh = Head(64)
        self.hm = Head(64)
        self.hl = Head(64)

    def forward(self, x):
        x = self.backbone(x)
        x = self.neck(x)

        p3 = self.hh(x.p3)
        p4 = self.hm(x.p4)
        p5 = self.hl(x.p5)

        return PyramidResult(p3, p4, p5)

In [70]:
fpn = FPN()

inp = torch.rand(1, 3, 640, 640)

out = fpn(inp)

print(out.p3.shape)

torch.Size([1, 24, 80, 80])
